归档说明（2026-11-05）

本 notebook 保留已完成的 5000 题采样、筛选与统计输出，作为运行记录。
最终 DPO 训练文件：outputs/medqa5000_L4_seed42_qpb2/selection_seed42/dpo.jsonl（3322 对）。

主工程仅保留 DPO 训练数据及来源审计。原始 samples、SFT 导出、运行日志、详细报告及硬件测试输出已移动到工程上一级的“备份文件夹/cleanup_20261105/medqa-calibration/”，相对路径不变。
重新执行本 notebook 的 build/完整性统计需要先恢复 samples 和 sft.jsonl；本 notebook 的执行配置仍为 Colab 路径，不应在本地直接点击全部运行。
32 题硬件对比单元已从当前 notebook 移除，其代码和输出保存在备份中。


# DPO / SFT 数据准备
调用 `train/dpo/prepare.py`，完成八回答采样、筛选、配对及统计。Notebook 输出会记录运行日志、表格和报告；执行后请保存 notebook。

- 本地：选择 **DL_Final** Python 内核。Colab：选择 GPU runtime，准备工程及其依赖；如使用 Drive，先在下面挂载。
- 多题批处理版本默认先运行 **8 题**。正式运行设置 `N_QUESTIONS = 0`，并使用新的输出目录。
- 每题固定采样 8 次。SFT 与 DPO 使用同一个正确回答；无法解析或截断的回答不进入配对。
- 本 notebook 不执行模型训练。两种来源均使用工程固定的训练划分。

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 1. 配置
`SOURCE="local"`：本地模型和 JSONL；`SOURCE="hf"`：HF 模型及 MedQA train。
本地默认目录对应当前工程的上一级；Colab 中请设置 `REPO_DIR`，并将 `OUTPUT_DIR` 指向已挂载的 Drive。
断点续跑必须保持采样参数和来源一致；换实验时使用新目录。`SELECTION_SEED` 与训练时的随机种子不同。

In [2]:
from pathlib import Path
import sys,json,os,subprocess,time,shutil
from datetime import datetime,timezone
import pandas as pd
from IPython.display import display,Markdown,HTML
DRIVE_REPO=Path('/content/drive/MyDrive/medqa-calibration')
assert (DRIVE_REPO/'train/dpo/prepare.py').is_file()
REPO_DIR=Path('/content/medqa-full')
REPO_DIR.mkdir(exist_ok=True)
for folder in ['train','data','eval']:
    shutil.copytree(DRIVE_REPO/folder,REPO_DIR/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.6','datasets','accelerate','pandas','safetensors'])
import torch
GPU_NAME=torch.cuda.get_device_name(0)
GPU_TAG='L4'
assert 'L4' in GPU_NAME,GPU_NAME
SOURCE='hf'
N_QUESTIONS=5000
QUESTIONS_PER_BATCH=2
SAMPLES_PER_BATCH=8
MAX_NEW_TOKENS=1024
TEMPERATURE=1.0
TOP_P=1.0
SAMPLING_SEED=42
SELECTION_SEED=42
QUANTIZATION='none'
MODEL_REVISION='main'
DATASET_REVISION='main'
HF_MODEL='Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR=DRIVE_REPO/'outputs'/'medqa5000_L4_seed42_qpb2'
print('GPU:',GPU_NAME,'| questions:',N_QUESTIONS,'| output:',OUTPUT_DIR)


GPU: NVIDIA L4 | questions: 5000 | output: /content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2


## 2. 环境检查及日志工具
使用当前 notebook 内核的 Python 调用模块。无 CUDA 时停止，避免意外在 CPU 上执行长时间采样。日志逐行显示，并保存在输出目录中。

In [3]:
import torch, threading
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0), '| BF16:', torch.cuda.is_bf16_supported())
MODULE = 'train.dpo.prepare'
def run_stage(stage, arguments):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log_path = OUTPUT_DIR / f'{stage}_{stamp}.log'
    command = [sys.executable,'-u','-m',MODULE,stage,*map(str,arguments)]
    print('Command:',command,flush=True)
    usage=[]
    stop=threading.Event()
    def monitor():
        while not stop.is_set():
            try:
                line=subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,utilization.gpu','--format=csv,noheader,nounits'],text=True).strip().splitlines()[0]
                mem,util=map(float,line.split(','))
                usage.append((mem,util))
            except Exception:
                pass
            stop.wait(1)
    watcher=threading.Thread(target=monitor,daemon=True)
    watcher.start()
    started=time.perf_counter()
    env=dict(os.environ,PYTHONIOENCODING='utf-8',PYTHONUNBUFFERED='1')
    try:
        with log_path.open('w',encoding='utf-8') as log:
            proc=subprocess.Popen(command,cwd=REPO_DIR,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace')
            try:
                for line in proc.stdout:
                    print(line,end='',flush=True)
                    log.write(line); log.flush()
                rc=proc.wait()
            except BaseException:
                proc.terminate(); proc.wait(); raise
            finally:
                proc.stdout.close()
    finally:
        stop.set(); watcher.join(timeout=3)
    elapsed=time.perf_counter()-started
    record={'stage':stage,'gpu':GPU_NAME,'command':command,'return_code':rc,'seconds':round(elapsed,2),'peak_gpu_memory_mib':max((x[0] for x in usage),default=None),'mean_gpu_util_percent':sum(x[1] for x in usage)/len(usage) if usage else None,'log':str(log_path)}
    (OUTPUT_DIR/f'{stage}_{stamp}.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
    print('BENCHMARK',json.dumps(record),flush=True)
    if rc: raise RuntimeError(f'{stage} failed: {rc}')
    return record


PyTorch: 2.11.0+cu130 | CUDA: 13.0
GPU: NVIDIA L4 | BF16: True


## 3. 八回答采样
此单元格会实际加载模型并生成回答。`N_QUESTIONS=0` 会开始全量采样；执行前确认配置。已完成的题目会跳过。

`QUESTIONS_PER_BATCH × SAMPLES_PER_BATCH` 是一次生成的并行回答数；每题总计仍为八个回答。新批处理版本必须使用新输出目录。改变任一批大小会改变随机采样结果，不能混用旧结果。部分批次中断后会重新计算整批，但保留已有文件，仅补写缺失题目。


In [4]:
from huggingface_hub import snapshot_download
from datasets import load_dataset
print('Preparing model and dataset cache...',flush=True)
snapshot_download(HF_MODEL,revision=MODEL_REVISION,allow_patterns=['*.json','*.safetensors','*.txt'])
load_dataset('GBaker/MedQA-USMLE-4-options',split='train',revision=DATASET_REVISION)
assert N_QUESTIONS==5000 and 'L4' in GPU_NAME
sample_args=['--source',SOURCE,'--n_questions',N_QUESTIONS,'--questions_per_batch',QUESTIONS_PER_BATCH,'--samples_per_batch',SAMPLES_PER_BATCH,'--max_new_tokens',MAX_NEW_TOKENS,'--temperature',TEMPERATURE,'--top_p',TOP_P,'--seed',SAMPLING_SEED,'--quantization',QUANTIZATION,'--output_dir',OUTPUT_DIR,'--model',HF_MODEL,'--revision',MODEL_REVISION,'--dataset_revision',DATASET_REVISION]
print('Resume enabled; existing completed questions are skipped. Output:',OUTPUT_DIR)
sample_run=run_stage('sample',sample_args)


Preparing model and dataset cache...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (8 outputs left out)

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

流式输出内容被截断，只能显示最后 5000 行内容。
2/5000 saved medqa4-1754.json
3/5000 saved medqa4-3718.json
4/5000 saved medqa4-9107.json
5/5000 saved medqa4-1838.json
6/5000 saved medqa4-4147.json
7/5000 saved medqa4-3631.json
8/5000 saved medqa4-2397.json
9/5000 saved medqa4-9469.json
10/5000 saved medqa4-9045.json
11/5000 saved medqa4-7637.json
12/5000 saved medqa4-5978.json
13/5000 saved medqa4-3925.json
14/5000 saved medqa4-9953.json
15/5000 saved medqa4-286.json
16/5000 saved medqa4-4202.json
17/5000 saved medqa4-3594.json
18/5000 saved medqa4-8370.json
19/5000 saved medqa4-7085.json
20/5000 saved medqa4-9654.json
21/5000 saved medqa4-5122.json
22/5000 saved medqa4-9425.json
23/5000 saved medqa4-1854.json
24/5000 saved medqa4-4833.json
25/5000 saved medqa4-6667.json
26/5000 saved medqa4-3572.json
27/5000 saved medqa4-7839.json
28/5000 saved medqa4-8768.json
29/5000 saved medqa4-6952.json
30/5000 saved medqa4-9525.json
31/5000 saved medqa4-2295.json
32/5000 saved medqa4-9781.json
33/5000 saved medqa4-

## 4. 筛选并导出 SFT / DPO 数据
每道题按固定种子选择一个正确回答和一个错误回答。此步骤不加载模型。

In [5]:
build_run = run_stage('build', ['--output_dir', OUTPUT_DIR, '--selection_seed', SELECTION_SEED])

Command: ['/usr/bin/python3', '-u', '-m', 'train.dpo.prepare', 'build', '--output_dir', '/content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2', '--selection_seed', '42']
Retained 3322/5000 questions. Output: /content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2/selection_seed42
BENCHMARK {"stage": "build", "gpu": "NVIDIA L4", "command": ["/usr/bin/python3", "-u", "-m", "train.dpo.prepare", "build", "--output_dir", "/content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2", "--selection_seed", "42"], "return_code": 0, "seconds": 11.69, "peak_gpu_memory_mib": 3.0, "mean_gpu_util_percent": 2.3333333333333335, "log": "/content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2/build_20261105T022743719565Z.log"}


## 5. 完整性检查、统计与可视化
直接读取保存的结果，能够在重新打开 notebook 后复查。统计将区分原始答案正确与满足配对条件的回答。

In [6]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from train.grpo.rewards import extract_answer
from data.medqa_split import load_split

manifest = read_json(OUTPUT_DIR / 'sampling_manifest.json')
config = manifest['config']
selected_dir = OUTPUT_DIR / f'selection_seed{SELECTION_SEED}'
audit = read_jsonl(selected_dir / 'selection_audit.jsonl')
sft = read_jsonl(selected_dir / 'sft.jsonl')
dpo = read_jsonl(selected_dir / 'dpo.jsonl')
retained = read_jsonl(selected_dir / 'retained_questions.jsonl')
records = [read_json(OUTPUT_DIR / 'samples' / f'medqa4-{i}.json') for i in config['train_indices']]
by_id = {r['question_id']: r for r in records}
audit_by_id = {a['question_id']: a for a in audit}
split = load_split()
assert len(by_id) == len(records) == len(audit_by_id) == len(audit)
assert set(by_id) == set(audit_by_id)
assert set(config['train_indices']).issubset(split['train_indices'])
assert not set(config['train_indices']).intersection(split['val_indices'])
assert all(len(r['completions']) == 8 for r in records)
assert len(sft) == len(dpo) == len(retained) == sum(a['retained'] for a in audit)
for s, d, kept in zip(sft, dpo, retained):
    r = by_id[kept['question_id']]
    a = audit_by_id[r['question_id']]
    assert s['prompt'] == d['prompt'] == r['prompt']
    assert s['completion'] == d['chosen']
    assert extract_answer(d['chosen'][0]['content']) == r['gold']
    assert extract_answer(d['rejected'][0]['content']) in set('ABCD') - {r['gold']}
    for key in ('chosen', 'rejected'):
        c = r['completions'][a[f'{key}_sample_id']]
        assert not c['truncated'] and c['text'] == d[key][0]['content']

rows = []
for r in records:
    completions = r['completions']
    valid = [c for c in completions if not c['truncated'] and extract_answer(c['text']) in set('ABCD')]
    nc = sum(extract_answer(c['text']) == r['gold'] for c in valid)
    nw = len(valid) - nc
    a = audit_by_id[r['question_id']]
    assert (nc, nw, bool(nc and nw)) == (a['n_correct'], a['n_wrong'], a['retained'])
    rows.append({'question_id': r['question_id'], 'gold': r['gold'],
                 'eligible_correct': nc, 'eligible_wrong': nw,
                 'unparsed': sum(extract_answer(c['text']) is None for c in completions),
                 'truncated': sum(c['truncated'] for c in completions),
                 'retained': a['retained']})
per_question = pd.DataFrame(rows)
total = len(records)*8
summary = {
    'candidate_questions': len(records), 'generated_answers': total,
    'retained_questions': len(retained), 'excluded_questions': len(records)-len(retained),
    'retention_rate': len(retained)/len(records) if records else 0,
    'eligible_correct_answers': int(per_question['eligible_correct'].sum()),
    'eligible_wrong_answers': int(per_question['eligible_wrong'].sum()),
    'unparsed_answers': int(per_question['unparsed'].sum()),
    'truncated_answers': int(per_question['truncated'].sum()),
    'sft_rows': len(sft), 'dpo_pairs': len(dpo), 'validation_passed': True,
}
display(pd.DataFrame(summary.items(), columns=['指标', '值']))
display(per_question.head(30))
print('完整逐题统计将保存为 CSV；无法解析与截断数量可能重叠。')
if not retained:
    print('没有题目满足正负回答配对条件；导出为空，不可用于训练。')

,指标,值
0,candidate_questions,5000
1,generated_answers,40000
2,retained_questions,3322
3,excluded_questions,1678
4,retention_rate,0.6644
5,eligible_correct_answers,19528
6,eligible_wrong_answers,20362
7,unparsed_answers,109
8,truncated_answers,2
9,sft_rows,3322


,question_id,gold,eligible_correct,eligible_wrong,unparsed,truncated,retained
0,medqa4-2622,C,0,8,0,0,False
1,medqa4-1754,B,4,4,0,0,True
2,medqa4-3718,C,5,3,0,0,True
3,medqa4-9107,D,8,0,0,0,False
4,medqa4-1838,D,3,5,0,0,True
5,medqa4-4147,C,8,0,0,0,False
6,medqa4-3631,B,7,1,0,0,True
7,medqa4-2397,C,2,6,0,0,True
8,medqa4-9469,C,6,2,0,0,True
9,medqa4-9045,C,7,1,0,0,True


完整逐题统计将保存为 CSV；无法解析与截断数量可能重叠。


## 6. 汇报与保存
保存汇总 JSON、逐题 CSV 和可独立打开的 HTML 报告。图表不依赖额外绘图库。

In [7]:
import html
bars = []
for label, count, color in [('保留题目', len(retained), '#2a9d8f'),
                             ('排除题目', len(records)-len(retained), '#e9a23b')]:
    pct = count / max(len(records), 1) * 100
    bars.append(f'<p>{label}：{count}（{pct:.1f}%）</p><div style="background:#eee;height:22px"><div style="background:{color};width:{pct}%;height:22px"></div></div>')
report_html = ('<!doctype html><html><head><meta charset="utf-8"><title>DPO 数据准备报告</title></head>'
               '<body style="font-family:sans-serif;max-width:1000px;margin:30px auto;padding:20px">'
               '<h1>SFT / DPO 数据生成报告</h1>'
               f'<p>输出目录：{html.escape(str(OUTPUT_DIR))}</p>'
               '<p>每题生成 8 个回答；仅使用非截断、可解析的正确和错误回答构建偏好对。</p>'
               + ''.join(bars) + '<h2>统计</h2>'
               + pd.DataFrame(summary.items(), columns=['指标','值']).to_html(index=False)
               + '<h2>逐题详情</h2>' + per_question.to_html(index=False)
               + '<h2>采样配置</h2><pre>' + html.escape(json.dumps(config, ensure_ascii=False, indent=2))
               + '</pre></body></html>')
report_dir = selected_dir / 'report'
report_dir.mkdir(exist_ok=True)
(report_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
per_question.to_csv(report_dir / 'per_question.csv', index=False, encoding='utf-8-sig')
(report_dir / 'report.html').write_text(report_html, encoding='utf-8')
display(HTML(''.join(bars)))
display(Markdown(f"完成 **{len(records)} 题 / {total} 个回答**，保留 **{len(retained)} 题**，导出 **{len(sft)} 条 SFT 数据、{len(dpo)} 对 DPO 数据**。"))
print('报告目录:', report_dir)
print('SFT:', selected_dir / 'sft.jsonl')
print('DPO:', selected_dir / 'dpo.jsonl')
print('请保存 notebook，以保留当前执行输出。')

完成 **5000 题 / 40000 个回答**，保留 **3322 题**，导出 **3322 条 SFT 数据、3322 对 DPO 数据**。

报告目录: /content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2/selection_seed42/report
SFT: /content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2/selection_seed42/sft.jsonl
DPO: /content/drive/MyDrive/medqa-calibration/outputs/medqa5000_L4_seed42_qpb2/selection_seed42/dpo.jsonl
请保存 notebook，以保留当前执行输出。
